In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install rdkit==2024.9.5
!pip install torch_geometric==2.5.3

In [2]:
import os
import sys
doc_name = "/content/drive/MyDrive/BHRGNN-Code"
sys.path.append(doc_name)
import torch
from torch_geometric.nn import GATConv
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import global_mean_pool as gap
from torch_geometric.nn import global_max_pool as gmp
from torch_geometric.loader import DataLoader
from torch_geometric.data import Data
from utils.BRG import *
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
import random
import numpy as np
import csv
import datetime
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
import seaborn as sns
import pandas as pd
from matplotlib.ticker import FuncFormatter

# BHRGNN Model Framework

In [16]:
class Net(nn.Module):
    def __init__(self, in_channels):
        super(Net, self).__init__()
        self.conv1 = GATConv(in_channels, 512, heads=2)
        self.norm1 = nn.BatchNorm1d(1024)
        self.conv2 = GATConv(1024, 256, heads=2)
        self.norm2 = nn.BatchNorm1d(512)

        self.lin1 = nn.Linear(3072, 1024)
        self.lin2 = nn.Linear(1024, 512)
        self.lin3 = nn.Linear(512, 256)
        self.lin4 = nn.Linear(256, 128)
        self.lin5 = nn.Linear(128, 1)
        self.bn1 = nn.BatchNorm1d(3072)
        self.bn2 = nn.BatchNorm1d(1024)
        self.bn3 = nn.BatchNorm1d(512)
        self.bn4 = nn.BatchNorm1d(256)

    def get_mask_edge_indices(self, edge_index, batch, mask_edge_positions=None):
        unique_batches, counts = torch.unique(batch[edge_index[0]], return_counts=True)
        mask_indices = []
        start_idx = 0

        for i, count in enumerate(counts):
            for mask_edge_position in mask_edge_positions:
                if mask_edge_position < count:
                    mask_indices.append(start_idx + mask_edge_position)
            start_idx += count

        return torch.tensor(mask_indices, dtype=torch.long, device=edge_index.device)

    def mask_edges(self, edge_index, mask_indices):
        masked_edge_index = edge_index[:, mask_indices]
        mask = torch.ones(edge_index.shape[1], dtype=torch.bool, device=edge_index.device)
        mask[mask_indices] = False
        edge_index = edge_index[:, mask]
        return edge_index, masked_edge_index

    def forward(self, data, mask_node=None,  mask_node_feature=None, mask_edge=None):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        # Masking Edge
        if mask_edge is not None:
            if isinstance(mask_edge, int):
                mask_edge = [mask_edge]
            mask_indices = self.get_mask_edge_indices(edge_index, batch, mask_edge)
            edge_index, masked_edge_index = self.mask_edges(edge_index, mask_indices)

        # Masking Node
        mask = 0
        if mask_node is not None:
            num_nodes_per_graph = data.num_nodes // len(torch.unique(batch))
            mask_indices = [i * num_nodes_per_graph + mask_node for i in range(len(torch.unique(batch)))]
            mask_indices = torch.tensor(mask_indices, dtype=torch.long, device=x.device)

            if mask_node_feature is not None:
                x[mask_indices, mask_node_feature] = mask
            else:
                x[mask_indices, :] = x[mask_indices, :] * mask

        x = self.conv1(x, edge_index)
        x = self.norm1(x)
        x1 = F.relu(x)

        x = self.conv2(x1, edge_index)
        x = self.norm2(x)
        x2 = F.relu(x)

        x = torch.cat([x1, x2], 1)

        x_mean = gap(x, batch=batch)
        x_max = gmp(x, batch=batch)
        x = torch.cat([x_mean, x_max], 1)

        x = x.view(x.shape[0], -1)

        x = self.bn1(x)
        x = self.lin1(x)
        x = nn.Dropout(p=0.5)(x)
        x = nn.ReLU()(x)

        x = self.bn2(x)
        x = self.lin2(x)
        x = nn.ReLU()(x)

        x = self.bn3(x)
        x = self.lin3(x)
        x = nn.ReLU()(x)

        x = self.bn4(x)
        x = self.lin4(x)
        x = nn.ReLU()(x)

        x = self.lin5(x)

        return x.squeeze()

#  Model Training

In [ ]:
def train_model(connection_type:str,
         features,
         edge_index:list,
         report_addr:str,
         rs:int=42,
         lr:float=1e-4,
         batch_size:int=64,
         num_epochs:int=2000,
         reture_metrics:bool=True,
         mode:str="test"
         ):
  assert mode in ("test", "opt"), f"mode must be 'test' or 'opt', got {mode}"

  # random state
  random.seed(rs)
  torch.manual_seed(rs)

  # Dataset
  datas = create_all_graphs(features, edge_index)
  dataset = CustomGraphDataset(datas, labels)

  # dataset splitting
  train_size = int(0.7 * len(dataset))
  val_size = len(dataset) - train_size
  train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

  # DataLoader
  train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
  val_loader = DataLoader(val_dataset, batch_size=val_size)
  print(f'{connection_type} trainning data generated.')

  # model initialization
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  model = Net(100).to(device)
  opti = torch.optim.Adam(model.parameters(), lr=lr)
  criterion = torch.nn.MSELoss()

  # report
  if mode == "opt":
    dir_path = "%s/model_training/opt-BHRGNN_%s_lr=%s_bs=%s_%s" % (report_addr, connection_type, lr, batch_size, datetime.datetime.now())
  if mode == "test":
    dir_path = "%s/model_training/test-BHRGNN_%s_rs=%s_%s" % (report_addr, connection_type, rs, datetime.datetime.now())
  os.mkdir("%s" % dir_path)
  metrics_path = '%s/metrics.csv' % dir_path

  # evaluarion metrics
  metrics_names = ['Epoch', 'Train_Loss', 'Test_Loss', 'Train_RMSE', 'Test_RMSE', 'Train_R2', 'Test_R2', 'Train_MAE', 'Test_MAE']
  train_losses = []
  test_losses = []
  maes_train = []
  rmses_train = []
  r2s_train = []
  maes_test = []
  rmses_test = []
  r2s_test = []

  with open(metrics_path, 'w', newline='') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=metrics_names)
    writer.writeheader()

  print(f'Traning GNN for {connection_type}:')

  # train
  if mode == "test":
    desc = f"Epochs for {connection_type} Model rs={rs}"
  if mode == "opt":
    desc = f"Epochs for {connection_type} Model lr={lr}, bs={batch_size}"
  for epoch in tqdm(range(num_epochs), desc=desc):
    model.train()
    running_loss_train = 0.0
    all_preds_train = []
    all_targets_train = []

    for data in train_loader:
        out = model(data[0].to(device))
        loss = criterion(out, data[1].to(device))
        opti.zero_grad()
        loss.backward()
        opti.step()
        running_loss_train += loss.item()

        # save predicted value and ground truth for evaluation metrics calculation
        all_preds_train.append(out.detach().cpu().numpy())
        all_targets_train.append(data[1].detach().cpu().numpy())

    epoch_loss_train = running_loss_train / len(train_loader)

    # Convert to numpy
    all_preds_train = np.concatenate(all_preds_train)
    all_targets_train = np.concatenate(all_targets_train)

    # evaluation metrics for trainset
    mae_train = mean_absolute_error(all_targets_train, all_preds_train)
    rmse_train = np.sqrt(np.mean((all_preds_train - all_targets_train) ** 2))
    r2_train = r2_score(all_targets_train, all_preds_train)

    with torch.no_grad():
        running_loss_val = 0.0
        all_preds_val = []
        all_targets_val = []

        for data in val_loader:
            out = model(data[0].to(device))
            val_loss = criterion(out, data[1].to(device))
            running_loss_val += val_loss.item()

            # save predicted value and ground truth
            all_preds_val.append(out.detach().cpu().numpy())
            all_targets_val.append(data[1].detach().cpu().numpy())

        epoch_loss_val = running_loss_val / len(val_loader)
        test_losses.append(epoch_loss_val)

        # Convert to numpy
        all_preds_val = np.concatenate(all_preds_val)
        all_targets_val = np.concatenate(all_targets_val)

        # evaluation metrics for testset
        mae_val = mean_absolute_error(all_targets_val, all_preds_val)
        rmse_val = np.sqrt(np.mean((all_preds_val - all_targets_val) ** 2))
        r2_val = r2_score(all_targets_val, all_preds_val)

    # the record of evaluation metrics for this round
    train_losses.append(epoch_loss_train)
    maes_train.append(mae_train)
    rmses_train.append(rmse_train)
    r2s_train.append(r2_train)
    maes_test.append(mae_val)
    rmses_test.append(rmse_val)
    r2s_test.append(r2_val)

    with open(metrics_path, 'a', newline='') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=metrics_names)
        writer.writerow({
            'Epoch': epoch + 1,
            'Train_Loss': epoch_loss_train,
            'Test_Loss': epoch_loss_val,
            'Train_RMSE': rmse_train,
            'Test_RMSE': rmse_val,
            'Train_R2': r2_train,
            'Test_R2': r2_val,
            'Train_MAE': mae_train,
            'Test_MAE': mae_val
        })
  print(f'Finish training for {connection_type} B-H graph')
  # best performance
  best_index = r2s_test.index(max(r2s_test))

  # Plot of the model performance
  # R2
  plt.figure(figsize=(8, 5), dpi=200)
  plt.plot(range(1, num_epochs + 1), r2s_train, label='Trainset $R^2$')
  plt.plot(range(1, num_epochs + 1), r2s_test, label='Testset $R^2$', linestyle='--')
  plt.text(1000, 0.5, 'R$^2$=%.4f' % r2s_test[best_index], fontsize=14, va='center', ha='center')
  plt.title('Train/Test $R^2$ Over Epochs')
  plt.xlabel('Epoch')
  plt.ylabel('$R^2$')
  plt.legend()
  plt.savefig("%s/R2_performance.png" % dir_path)

  # MSE
  plt.figure(figsize=(8, 5), dpi=200)
  plt.plot(range(1, num_epochs + 1), train_losses, label='Trainset Loss')
  plt.plot(range(1, num_epochs + 1), test_losses, label='Testset Loss', linestyle='--')
  plt.text(1000, 0.05, 'MSE=%.4f' % test_losses[best_index], fontsize=14, va='center', ha='center')
  plt.title('Train/Test MSE Over Epochs')
  plt.xlabel('Epoch')
  plt.ylabel('MSE Loss')
  plt.legend()
  plt.savefig("%s/MSE_performance.png" % dir_path)

  # Output of the model best performance
  if reture_metrics:
    return {
        "train_R2":r2s_train[best_index],
        "train_RMSE":rmses_train[best_index],
        "train_MAE":maes_train[best_index],
        "test_R2":r2s_test[best_index],
        "test_RMSE":rmses_test[best_index],
        "test_MAE":maes_test[best_index],
        "model":model
    }

In [ ]:
# import data
files = ['Aryl_halide_new.csv', 'Product_new.csv', 'Base_new.csv', 'Ligand_new.csv', 'Additive_new.csv']
for i in range(len(files)):
  files[i] = doc_name + "/data/" + files[i]
features = load_and_preprocess_data(files)
labels = pd.read_csv('%s/data/Buchwald.csv' % doc_name).iloc[:, -1].values / 100

# Visualization of the connection type
Visualization = True
if Visualization:
  data=Data(edge_index=torch.tensor([[0,0,0,1,1,2,2,3,3,4,4,4,4,4],
                    [0,1,2,1,0,2,3,3,1,0,1,2,3,4]], dtype=torch.long))
  visualize_graph_data(data=data, num_nodes=5, connect_type="Buchwald-Hartwig Graph", custom_positions={0: (0, 1),1: (2, 1),2: (0.5, 0),3: (1.5, 0),4: (1, 2)}, node_labels={0: 'Aryl',1: ' Product',2: 'Base',3: 'Ligand',4: 'Additive'})

mode = "test" # test or opt
assert mode in ("test", "opt"), f"mode must be 'test' or 'opt', got {mode}"

rs_list = [1,2,3,4,42]
lr_list = [1e-3, 1e-4, 1e-5]
batch_size_list = [32, 64, 128]

if mode == "opt":
  for lr in lr_list:
    for bs in batch_size_list:
      # Evaluation metrics
      eval_metrics = np.zeros((1, 6))
      columns = ['train_R2', 'train_RMSE','train_MAE','test_R2','test_RMSE','test_MAE']
      index = ["lr=%s_bs=%s" % (lr, bs)]
      eval_metrics = pd.DataFrame(eval_metrics, columns=columns, index=index)
      # training
      metrics=train_model(connection_type="Buchwald-Hartwig Graph",
            features=features,
            edge_index = torch.tensor([[0,0,0,1,1,2,2,3,3,4,4,4,4,4],
                        [0,1,2,1,0,2,3,3,1,0,1,2,3,4]], dtype=torch.long),
            report_addr = "%s" % doc_name,
            rs = 42,
            lr = lr,
            batch_size = bs,
            num_epochs = 2000,
            reture_metrics = True,
            mode=mode
            )
      for name in eval_metrics.columns:
        eval_metrics.loc["lr=%s_bs=%s" % (lr, bs)][name] = metrics[name]

      # Evaluation metrics report
      eval_metrics.to_csv("%s/model_training/opt-BHRGNN-%s_lr=%s_bs=%s_%s.csv" % (doc_name, "Buchwald-Hartwig Graph", lr, bs, datetime.datetime.now()))
      print(eval_metrics)

if mode == "test":
  # model
  model=[]

  # Evaluation metrics
  eval_metrics = np.zeros((1+len(rs_list), 6))
  columns = ['train_R2', 'train_RMSE','train_MAE','test_R2','test_RMSE','test_MAE']
  index = []
  for rs in rs_list:
    index.append("%s" % rs)
  index.append("avg±std")
  eval_metrics = pd.DataFrame(eval_metrics, columns=columns, index=index)

  for m in range(len(rs_list)):
    metrics=train_model(connection_type="Buchwald-Hartwig Graph",
          features=features,
          edge_index = torch.tensor([[0,0,0,1,1,2,2,3,3,4,4,4,4,4],
                      [0,1,2,1,0,2,3,3,1,0,1,2,3,4]], dtype=torch.long),
          report_addr = "%s" % doc_name,
          rs = rs_list[m],
          lr = 1e-3,
          batch_size = 128,
          num_epochs = 2000,
          reture_metrics = True,
          mode=mode
          )
    for name in eval_metrics.columns:
      eval_metrics.loc["%s" % rs_list[m]][name] = metrics[name]
    model=metrics["model"]

  # Evaluation metrics report
  for i in range(len(rs_list), eval_metrics.shape[0], len(rs_list)+1):
    for j in range(eval_metrics.shape[1]):
      eval_metrics.iloc[i,j] = "%.4f ± %.4f" % (eval_metrics.iloc[i-len(rs_list):i-1,j].mean(), eval_metrics.iloc[i-len(rs_list):i-1,j].std())
  eval_metrics.to_csv("%s/model_training/test-BHRGNN-%s_%s.csv" % (doc_name, "Buchwald-Hartwig Graph", datetime.datetime.now()))
  print(eval_metrics)

# Model explanation

### Edge Masking

In [ ]:
# Set seed
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Dataloader
# import data
files = ['Aryl_halide_new.csv', 'Product_new.csv', 'Base_new.csv', 'Ligand_new.csv', 'Additive_new.csv']
for i in range(len(files)):
  files[i] = doc_name + "/data/" + files[i]
features = load_and_preprocess_data(files)
labels = pd.read_csv('%s/data/Buchwald.csv' % doc_name).iloc[:, -1].values / 100
datas = create_all_graphs(features, edge_index=torch.tensor([[0,0,0,1,1,2,2,3,3,4,4,4,4,4],
                    [0,1,2,1,0,2,3,3,1,0,1,2,3,4]], dtype=torch.long))
dataset = CustomGraphDataset(datas, labels)
dataset_loader = DataLoader(dataset, batch_size=512, shuffle=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# edges to be masked
mask_edge_list = [[1,4], [2], [6], [8], [9], [10], [11], [12]]
importance_scores = []

for mask_edge in mask_edge_list:
    all_masked_outputs = []
    all_labels = []

    with torch.no_grad():
        for data in dataset_loader:
            inputs, labels = data
            masked_output = model(inputs.to(device), mask_edge=mask_edge)
            all_masked_outputs.append(masked_output.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    all_masked_outputs = np.concatenate(all_masked_outputs)
    all_labels = np.concatenate(all_labels)

    # RMSE Calculation
    squared_differences = (all_masked_outputs - all_labels) ** 2
    mse = np.mean(squared_differences)
    importance_score = np.sqrt(mse)
    importance_scores.append(importance_score)
    print(f"Importance Score (RMSE): {importance_score}")

### Node Masking

In [ ]:
mask_node_index = range(5)
importance_scores = []

for mask_node in mask_node_index:
    all_masked_outputs = []
    all_labels = []

    with torch.no_grad():
        for data in dataset_loader:
            inputs, labels = data
            masked_output = model(inputs.to(device), mask_node=mask_node)
            all_masked_outputs.append(masked_output.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    all_masked_outputs = np.concatenate(all_masked_outputs)
    all_labels = np.concatenate(all_labels)

    # RMSE Calculation
    squared_differences = (all_masked_outputs - all_labels) ** 2
    mse = np.mean(squared_differences)
    importance_score = np.sqrt(mse)
    importance_scores.append(importance_score)
    print(f"Importance Score (RMSE): {importance_score}")

In [ ]:
df = pd.DataFrame(columns=['Node', 'Importance'])
df['Node'] = [i for i in range(5)]
cpd_num = [15, 15, 3, 4, 22]
ip_score = importance_scores
for i in range(len(ip_score)):
  ip_score[i] = ip_score[i] / cpd_num[i]
df['Importance'] = ip_score
df['Component'] = ['Aryl', 'Product', 'Base', 'Ligand', 'Additive']

sns.set(style="whitegrid")
plt.figure(figsize=(5, 5), dpi=300)

ax = sns.barplot(
    x='Node',
    y='Importance',
    hue='Component',
    data=df,
    palette='viridis',
    edgecolor='black',
    legend=False
)

plt.title('Node Importance Score Comparison', fontsize=16)
plt.ylabel('Importance Score', fontsize=14)
plt.xlabel('')

ax.set_xticks(range(len(df)))
ax.set_xticklabels(['Aryl', 'Product', 'Base', 'Ligand', 'Additive'], fontsize=14)

ax.patch.set_edgecolor('black')
ax.patch.set_linewidth(2)
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{x:.2f}'))
plt.yticks([])
plt.savefig("%s/model_training/Node_Importance.png" % doc_name)
plt.tight_layout()
plt.show()

### Node Features Masking

In [ ]:
from tqdm import tqdm

mask_node_index = 4
mask_node_features = range(100)
importance_scores = []

for mask_node_feature in tqdm(mask_node_features):
    all_masked_outputs = []
    all_labels = []

    with torch.no_grad():
        for data in dataset_loader:
            inputs, labels = data
            masked_output = model(inputs.to(device), mask_node=mask_node_index, mask_node_feature=mask_node_feature)
            all_masked_outputs.append(masked_output.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    all_masked_outputs = np.concatenate(all_masked_outputs)
    all_labels = np.concatenate(all_labels)

    # RMSE Calculation
    squared_differences = (all_masked_outputs - all_labels) ** 2
    mse = np.mean(squared_differences)
    importance_score = np.sqrt(mse)
    importance_scores.append(importance_score)

print('Node Features Masking Finished')

# Node Permutation in BHR

In [ ]:
mode = "test" # test or opt
rs_list = [1,2,3,4,42]

# import data
BHR_dict = {"BHR1":['Product_new.csv', 'Aryl_halide_new.csv', 'Base_new.csv', 'Ligand_new.csv', 'Additive_new.csv'],
            "BHR2":['Aryl_halide_new.csv', 'Product_new.csv', 'Additive_new.csv', 'Ligand_new.csv', 'Base_new.csv'],
            "BHR3":['Aryl_halide_new.csv', 'Product_new.csv', 'Ligand_new.csv', 'Base_new.csv', 'Additive_new.csv'],
            "BHR4":['Aryl_halide_new.csv', 'Product_new.csv', 'Base_new.csv', 'Additive_new.csv', 'Ligand_new.csv']}


for key in BHR_dict:
  files = BHR_dict[key]
  for i in range(len(files)):
    files[i] = doc_name + "/data/" + files[i]
  features = load_and_preprocess_data(files)
  labels = pd.read_csv('%s/data/Buchwald.csv' % doc_name).iloc[:, -1].values / 100

  Visualization = True
  if Visualization:
    data=Data(edge_index=torch.tensor([[0,0,0,1,1,2,2,3,3,4,4,4,4,4],
                      [0,1,2,1,0,2,3,3,1,0,1,2,3,4]], dtype=torch.long))
    visualize_graph_data(data=data,
               num_nodes=5,
               connect_type="%s" % key,
               custom_positions={0: (0, 1),1: (2, 1),2: (0.5, 0),3: (1.5, 0),4: (1, 2)},
               node_labels={0: files[0].split('_')[0].split('/')[-1],
                            1:files[1].split('_')[0].split('/')[-1],
                            2:files[2].split('_')[0].split('/')[-1],
                            3:files[3].split('_')[0].split('/')[-1],
                            4:files[4].split('_')[0].split('/')[-1]},
               fig_addr="%s" % doc_name + "/model_training")


  # Evaluation metrics
  eval_metrics = np.zeros((1+len(rs_list), 6))
  columns = ['train_R2', 'train_RMSE','train_MAE','test_R2','test_RMSE','test_MAE']
  index = []
  for rs in rs_list:
    index.append("%s" % rs)
  index.append("avg±std")
  eval_metrics = pd.DataFrame(eval_metrics, columns=columns, index=index)

  for m in range(len(rs_list)):
    metrics=train_model(connection_type="%s" % key,
          features=features,
          edge_index = torch.tensor([[0,0,0,1,1,2,2,3,3,4,4,4,4,4],
                      [0,1,2,1,0,2,3,3,1,0,1,2,3,4]], dtype=torch.long),
          report_addr = "%s" % doc_name,
          rs = rs_list[m],
          lr = 1e-3,
          batch_size = 128,
          num_epochs = 2000,
          reture_metrics = True,
          mode=mode
          )
    for name in eval_metrics.columns:
      eval_metrics.loc["%s" % rs_list[m]][name] = metrics[name]

  # Evaluation metrics report
  for i in range(len(rs_list), eval_metrics.shape[0], len(rs_list)+1):
    for j in range(eval_metrics.shape[1]):
      eval_metrics.iloc[i,j] = "%.4f ± %.4f" % (eval_metrics.iloc[i-len(rs_list):i-1,j].mean(), eval_metrics.iloc[i-len(rs_list):i-1,j].std())
  eval_metrics.to_csv("%s/model_training/permutation-BHRGNN-%s_%s.csv" % (doc_name, "%s" % key, datetime.datetime.now()))
  print(eval_metrics)

# FingerPrint as Node feature

In [18]:
mode = "test" # test or opt
rs_list = [1,2,3,4,42]

# import data
for FP in os.listdir("%s/data/FPdata" % doc_name):
  # Initialize the files: list
  files=[None] * 5
  for FP_data in os.listdir("%s/data/FPdata/%s" % (doc_name, FP)):
    if "Aryl" in FP_data:
      files[0] = FP_data
    if "Product" in FP_data:
      files[1] = FP_data
    if "Base" in FP_data:
      files[2] = FP_data
    if "Ligand" in FP_data:
      files[3] = FP_data
    if "Additive" in FP_data:
      files[4] = FP_data

  for i in range(len(files)):
    files[i] = doc_name + "/data/FPdata/%s/" % FP + files[i]
  features = load_and_preprocess_data(files)
  labels = pd.read_csv('%s/data/Buchwald.csv' % doc_name).iloc[:, -1].values / 100

  Visualization = False
  if Visualization:
    data=Data(edge_index=torch.tensor([[0,0,0,1,1,2,2,3,3,4,4,4,4,4],
                      [0,1,2,1,0,2,3,3,1,0,1,2,3,4]], dtype=torch.long))
    visualize_graph_data(data=data,
               num_nodes=5,
               connect_type="%s" % FP,
               custom_positions={0: (0, 1),1: (2, 1),2: (0.5, 0),3: (1.5, 0),4: (1, 2)},
               node_labels={0: files[0].split('_')[0].split('/')[-1],
                            1:files[1].split('_')[0].split('/')[-1],
                            2:files[2].split('_')[0].split('/')[-1],
                            3:files[3].split('_')[0].split('/')[-1],
                            4:files[4].split('_')[0].split('/')[-1]},
               fig_addr="%s" % doc_name + "/model_training")


  # Evaluation metrics
  eval_metrics = np.zeros((1+len(rs_list), 6))
  columns = ['train_R2', 'train_RMSE','train_MAE','test_R2','test_RMSE','test_MAE']
  index = []
  for rs in rs_list:
    index.append("%s" % rs)
  index.append("avg±std")
  eval_metrics = pd.DataFrame(eval_metrics, columns=columns, index=index)

  for m in range(len(rs_list)):
    metrics=train_model(connection_type="%s" % FP,
          features=features,
          edge_index = torch.tensor([[0,0,0,1,1,2,2,3,3,4,4,4,4,4],
                      [0,1,2,1,0,2,3,3,1,0,1,2,3,4]], dtype=torch.long),
          report_addr = "%s" % doc_name,
          rs = rs_list[m],
          lr = 1e-3,
          batch_size = 128,
          num_epochs = 2000,
          reture_metrics = True,
          mode=mode
          )
    for name in eval_metrics.columns:
      eval_metrics.loc["%s" % rs_list[m]][name] = metrics[name]

  # Evaluation metrics report
  for i in range(len(rs_list), eval_metrics.shape[0], len(rs_list)+1):
    for j in range(eval_metrics.shape[1]):
      eval_metrics.iloc[i,j] = "%.4f ± %.4f" % (eval_metrics.iloc[i-len(rs_list):i-1,j].mean(), eval_metrics.iloc[i-len(rs_list):i-1,j].std())
  eval_metrics.to_csv("%s/model_training/FP-BHRGNN-%s_%s.csv" % (doc_name, "%s" % FP, datetime.datetime.now()))
  print(eval_metrics)

Output hidden; open in https://colab.research.google.com to view.